# Design validation (interactive notebook)

This notebook is the interactive validation used during development. It runs the released
detector and SAM2 on the 30 hand-annotated slices and reports IoU, Dice and pixel accuracy
with visual overlays.

**The numbers reported in the paper come from `validation/tendon_eval.py`, not from this
notebook.** The two are not directly comparable: the `validation/` suite registers the
hand-drawn masks back onto the original image frames (`validation/gt_register.py`) and adds
per-fascicle instance matching, whereas this notebook aligns ground truth and prediction by
cropping each to its bounding box and resizing. See `validation/README.md` for the
authoritative results.


## Model Setup

In [ ]:
!pip install git+https://github.com/ultralytics/ultralytics.git
# Detector weights: the ones shipped with this repository.
!mkdir -p /content/weights
!wget -O /content/weights/fiberYOLO26Weights.pt https://github.com/Boyu-Zhang-UOI/INBRE-AI-Tendon/raw/main/fiberYOLO26Weights.pt
!pip install git+https://github.com/facebookresearch/segment-anything-2.git

## Upload Original Design Validation Images and Design Validation Images with Human Made Identification and Segmentation

In [ ]:
!pip install -q supervision jupyter_bbox_widget

import os
HOME = os.getcwd()
print("HOME:", HOME)

import cv2
import torch
import base64
import numpy as np
import supervision as sv

from pathlib import Path
import matplotlib.pyplot as plt
from supervision.assets import download_assets, VideoAssets
from torchvision import transforms
from PIL import Image
from jupyter_bbox_widget import BBoxWidget
from ultralytics import YOLO
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

IS_COLAB = True
if IS_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()

!mkdir -p {HOME}/checkpoints
!wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_tiny.pt -P {HOME}/checkpoints
!wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt -P {HOME}/checkpoints
!wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_base_plus.pt -P {HOME}/checkpoints
!wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt -P {HOME}/checkpoints

torch.autocast(device_type="cuda", dtype=torch.bfloat16).__enter__()

if torch.cuda.get_device_properties(0).major >= 8:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHECKPOINT = f"{HOME}/checkpoints/sam2_hiera_large.pt"
CONFIG = "sam2_hiera_l.yaml"
yolo_model = YOLO("/content/weights/fiberYOLO26Weights.pt")
sam2_model = build_sam2(CONFIG, CHECKPOINT)
predictor = SAM2ImagePredictor(sam2_model)

In [ ]:
!unzip /content/DesignValidationDataHandIdentification.zip -d /content
!unzip /content/DesignValidationDataHandSegmentation.zip -d /content
!unzip /content/ValidationDesignImagesOriginal30.zip -d /content

## Generate Pipeline Identifications and Segmentations

In [ ]:
modelData = []
filenames = sorted(os.listdir("/content/ValidationDesignImagesOriginal30"))
bbResults = yolo_model("/content/ValidationDesignImagesOriginal30")
for i in range(len(bbResults)):
  #I need to create list of the boxes to then use as input on the sam2 model each image in results needs to be run through
  #The whole system so we need to keep track of the boxes and the masks after both models run
  boxes = bbResults[i].boxes.xyxy.cpu().numpy()
  scores = bbResults[i].boxes.conf.cpu().numpy()
  fullBoxes = []
  for box, score in zip(boxes, scores):
    fullBoxes.append((box, float(score)))
  #print(fullBoxes)
  image = cv2.imread(f"/content/ValidationDesignImagesOriginal30/{filenames[i]}")
  predictor.set_image(image)
  segResults = predictor.predict(box=boxes, multimask_output=False)
  #print(segResults[0].shape)

  pipelineAttempt = {
      "image": filenames[i],
      "boxes": fullBoxes,
      "masks": segResults[0]
  }
  modelData.append(pipelineAttempt)


##Validation Generation

In [5]:
boxPath = "/content/Validation/Boxes"
maskPath = "/content/Validation/Masks"
if not os.path.exists(boxPath):
  os.makedirs(boxPath)
if not os.path.exists(maskPath):
  os.makedirs(maskPath)
for i, filename in enumerate(filenames):
  image = cv2.imread(f"/content/ValidationDesignImagesOriginal30/{filename}")
  boxes = modelData[i].get("boxes", [])
  for box, score in boxes:
    x1, y1, x2, y2 = box
    cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (255, 0, 0), 2)
  cv2.imwrite(os.path.join(boxPath, f"boxes{i}.jpg"), image)

for i, filename in enumerate(filenames):
  image = cv2.imread(f"/content/ValidationDesignImagesOriginal30/{filename}")
  masks = modelData[i].get("masks", [])
  if isinstance(masks, tuple):
    masks = masks[0]
  for mask in masks:
    while len(mask.shape) > 2:
      mask = mask[0]
    mask = (mask > 0).astype(np.uint8)
    colorMask = np.zeros_like(image)
    colorMask[:, :, 1] = mask * 255
    image = cv2.addWeighted(image, 1.0, colorMask, 0.5, 0)
  cv2.imwrite(os.path.join(maskPath, f"masks{i}.jpg"), image)


## Create Hand Drawn Visual

In [6]:
from re import X
labelFiles = sorted(os.listdir("/content/DesignValidationDataHandIdentification/labels"))
boxHandPath = "/content/Validation/BoxesHand"
#maskHandPath = "/content/Validation/MasksHand"
HandData = []
if not os.path.exists(boxHandPath):
  os.makedirs(boxHandPath)
#if not os.path.exists(maskHandPath):
#  os.makedirs(maskHandPath)
for i, filename in enumerate(filenames):
  hBoxes = []
  image = cv2.imread(f"/content/ValidationDesignImagesOriginal30/{filename}")
  imageWidth = image.shape[1]
  imageHeight = image.shape[0]
  filePath = os.path.join("/content/DesignValidationDataHandIdentification/labels/", labelFiles[i])
  with open(filePath, 'r', encoding='utf-8', errors='ignore') as labels:
    for j, label in enumerate(labels):
      x1 = ""
      y1 = ""
      x2 = ""
      y2 = ""
      spaceCount = 0;
      for char in label:
        if char == ' ':
          spaceCount = spaceCount + 1
          continue
        if spaceCount == 1:
          x1 = x1 + char
        elif spaceCount == 2:
          y1 = y1 + char
        elif spaceCount == 5:
          x2 = x2 + char
        elif spaceCount == 6:
          y2 = y2 + char
      x1 = float(x1) * imageWidth
      y1 = float(y1) * imageHeight
      x2 = float(x2) * imageWidth
      y2 = float(y2) * imageHeight
      boxAttempt = [x1, y1, x2, y2]
      boxArray = np.array(boxAttempt)
      hBoxes.append(boxArray)
      x1 = int(x1)
      y1 = int(y1)
      x2 = int(x2)
      y2 = int(y2)
      cv2.rectangle(image, (x1, y1), (x2, y2), (255, 0, 0), 2)
  hBoxes = np.array(hBoxes)
  handAttempt = {
    "image": filenames[i],
    "handBoxes": hBoxes
  }
  HandData.append(handAttempt)
  cv2.imwrite(os.path.join(boxHandPath, f"handBoxes{i}.jpg"), image)
#for i, filename in enumerate(filenames):
#  image = cv2.imread(f"/content/ValidationDesignImagesOriginal30/{filename}")


# Bounding Box AP calculation

In [ ]:
def BBIoU(boxA, boxB):
  xA = max(boxA[0], boxB[0])
  yA = max(boxA[1], boxB[1])
  xB = min(boxA[2], boxB[2])
  yB = min(boxA[3], boxB[3])
  intersect = max(0, xB - xA) * max(0, yB- yA)
  boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
  boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
  #we subtract intersect because we would be counting the overlap twice if not
  union = boxAArea + boxBArea - intersect
  return intersect / union if union > 0 else 0

def BBmatchPredToGT(groundTruth, predictions, threshold=0.5):
  detections = []
  truthBoxes = groundTruth.get("handBoxes", [])
  predBoxes = predictions.get("boxes", [])
  #print(truthBoxes)
  #print(predBoxes)
  #print(fileName)
  predBoxes = sorted(predBoxes, key=lambda x: x[1], reverse=True)
  usedTruth = set()

  for prediction in predBoxes:
    bestIoU = 0
    bestTruthIndex = -1
    for i, truth in enumerate(truthBoxes):
      if i in usedTruth:
        continue
      IoU = BBIoU(prediction[0], truth)
      if IoU > bestIoU:
        bestIoU = IoU
        bestTruthIndex = i

    if bestIoU >= threshold:
      usedTruth.add(bestTruthIndex)
      detections.append((prediction[1], 1)) #true positive
    else:
      detections.append((prediction[1], 0)) #false positive

  for i in range(len(truthBoxes)):
    if i not in usedTruth:
      detections.append((0, -1)) #false negative

  return detections

def AP(detections):

  detectionsNoFN = [x for x in detections if x[1] != -1]
  detectionsNoFN = sorted(detectionsNoFN, key=lambda x :x[0], reverse=True)

  TP = 0
  FP = 0
  precisions = []
  recalls = []
  totalGroundTruth = sum(1 for d in detections if d[1] in [1, -1])

  for detect in detectionsNoFN:
    if detect[1] == 1:
      TP += 1
    elif detect[1] == 0:
      FP += 1
    recall = TP / totalGroundTruth if totalGroundTruth > 0 else 0
    precision = TP / (TP + FP)
    recalls.append(recall)
    precisions.append(precision)

  ap = 0
  for i in range(1, len(recalls)):
    ap += (recalls[i] - recalls[i - 1]) * precisions[i]

  f1 = 2 * (precisions[len(precisions) - 1] * recalls[len(recalls) - 1]) / (precisions[len(precisions) - 1] + recalls[len(recalls) - 1])

  return ap, precisions, recalls, f1

finalList = []
aap = 0
aPrecision = 0
aRecall = 0
af1 = 0
for i in range(len(modelData)):
  thres = 0.5
  results = BBmatchPredToGT(HandData[i], modelData[i], thres)
  ap, precisions, recalls, f1 = AP(results)
  aap += ap
  aPrecision += precisions[len(precisions) - 1]
  aRecall += recalls[len(recalls) - 1]
  af1 += f1
  dataCache = {
      "image": modelData[i].get("image", []),
      "IoU Threshold": thres,
      "Recall": recalls[len(recalls) - 1],
      "Precision": precisions[len(precisions) - 1],
      "F1 Score": f1,
      "AP": ap
  }
  finalList.append(dataCache)
aap = aap / len(modelData)
aPrecision = aPrecision / len(modelData)
aRecall = aRecall / len(modelData)
af1 = af1 / len(modelData)
print("Average Average Precision")
print(aap)
print("Average Final Precision")
print(aPrecision)
print("Average Recall")
print(aRecall)
print("Average f1")
print(af1)

## Output Image

In [8]:
font = cv2.FONT_HERSHEY_SIMPLEX
font_scale = 1
color = (00, 255, 0) # Green color in BGR
thickness = 2

finalPath = "/content/Validation/Finals"
if not os.path.exists(finalPath):
  os.makedirs(finalPath)
for i, filename in enumerate(filenames):
  top_row = np.hstack((cv2.imread(f"/content/Validation/BoxesHand/handBoxes{i}.jpg"), cv2.imread(f"/content/Validation/Boxes/boxes{i}.jpg")))
  #bottom_row = np.hstack((cv2.imread(f"/content/ValidationDesignImagesOriginal30/{filename}"), cv2.imread(f"/content/Validation/Masks/masks{i}.jpg")))
  #grid = np.vstack((top_row, bottom_row))
  top_row = cv2.resize(top_row, (900, 450), interpolation=cv2.INTER_LINEAR)
  gridW = top_row.shape[1]
  gridH = top_row.shape[0]

  cv2.putText(top_row, "Ground Truth", (int(gridW / 7), int(gridH / 15)), font, font_scale, color, thickness, cv2.LINE_AA)
  cv2.putText(top_row, "Model Prediction", (int(gridW * 3 / 5), int(gridH / 15)), font, font_scale, color, thickness, cv2.LINE_AA)
  cv2.putText(top_row, f"IoU Threshold: {round(finalList[i].get("IoU Threshold", []), 2)}", (int(gridW * 1 / 60), int(gridH * 21 / 30)), font, font_scale, color, thickness, cv2.LINE_AA)
  cv2.putText(top_row, f"Recall: {round(finalList[i].get("Recall", []), 2)}", (int(gridW * 1 / 60), int(gridH * 23 / 30)), font, font_scale, color, thickness, cv2.LINE_AA)
  cv2.putText(top_row, f"Precision: {round(finalList[i].get("Precision", []), 2)}", (int(gridW * 1 / 60), int(gridH * 25 / 30)), font, font_scale, color, thickness, cv2.LINE_AA)
  cv2.putText(top_row, f"F1 Score: {round(finalList[i].get("F1 Score", []), 2)}", (int(gridW * 1 / 60), int(gridH * 27 / 30)), font, font_scale, color, thickness, cv2.LINE_AA)
  cv2.putText(top_row, f"AP: {round(finalList[i].get("AP", []), 2)}", (int(gridW * 1 / 60), int(gridH  * 29 / 30)), font, font_scale, color, thickness, cv2.LINE_AA)
  cv2.imwrite(os.path.join(finalPath, f"final{i}.jpg"), top_row)

In [ ]:
!zip -r '/content/ValidationFolder.zip' '/content/Validation/Finals'

In [ ]:
#Intersection over Union
#Dice Coefficient
#Pixel Accuracy
#mean + standard deviation of IoU/Dice

testMask = np.array(Image.open("/content/DesignValidationDataHandSegmentation/P21(1)_rec00000027.png"))
groundTruth = testMask[:, :, 3] > 0
groundTruth = groundTruth.astype(np.uint8)
masks = modelData[0].get("masks", [])
prediction = np.zeros(groundTruth.shape[:2], dtype=np.uint8)
for mask in masks:
  mask = mask[0]
  mask = cv2.resize(mask, (groundTruth.shape[1], groundTruth.shape[0]), interpolation=cv2.INTER_NEAREST)
  prediction = np.logical_or(prediction, mask).astype(np.uint8)
  print(groundTruth.shape)
  print(prediction.shape)
  prediction = cv2.resize(prediction, (groundTruth.shape[1], groundTruth.shape[0]), interpolation=cv2.INTER_NEAREST)

  #I need to either figure out how to put the ground truth in the right spot
  #Or blow out the prediciton to where the ground truth is
  #The first image she gave me has weird artifacts in the corners messing with its size
  #Test 1 for what ground truth looks like
  gmask = (groundTruth > 0).astype(np.uint8)
  image = cv2.imread("/content/ValidationDesignImagesOriginal30/P21(1)_rec00000025.jpeg")
  image = cv2.resize(image, (groundTruth.shape[1], groundTruth.shape[0]), interpolation=cv2.INTER_NEAREST)
  colorMask = np.zeros_like(image)
  colorMask[:, :, 1] = gmask * 255
  image = cv2.addWeighted(image, 1.0, colorMask, 0.5, 0)
  cv2.imwrite("/content/test1.png", image)

  #test 2 for what prediction looks like
  image = cv2.imread("/content/ValidationDesignImagesOriginal30/P21(1)_rec00000025.jpeg")
  image = cv2.resize(image, (groundTruth.shape[1], groundTruth.shape[0]), interpolation=cv2.INTER_NEAREST)
  colorMask = np.zeros_like(image)
  colorMask[:, :, 1] = prediction * 255
  image = cv2.addWeighted(image, 1.0, colorMask, 0.5, 0)
  cv2.imwrite("/content/test2.png", image)

  intersection = np.logical_and(prediction, groundTruth).sum()
  union = np.logical_or(prediction, groundTruth).sum()
  iou = intersection / union
  print(iou)

In [ ]:
all_ious = []
all_dice_coeffs = []
all_pixel_accuracies = []
import sys
np.set_printoptions(threshold=10)

# Assuming the ground truth mask files are named similarly to the image files but with .png extension
# For example, P21(1)_rec00000025.jpeg corresponds to P21(1)_rec00000025.png

if not os.path.exists("/content/Validation/Overlaps"):
  os.makedirs("/content/Validation/Overlaps")

for i, filename in enumerate(filenames):
  # Construct ground truth mask path
  gt_mask_filename = filename.replace(".jpeg", ".png")
  gt_mask_path = os.path.join("/content/DesignValidationDataHandSegmentation", gt_mask_filename)

  # Check if the ground truth mask file exists
  if not os.path.exists(gt_mask_path):
    print(f"Ground truth mask not found for {filename} at {gt_mask_path}. Skipping.")
    continue

  # Load ground truth mask
  gt_mask_raw = np.array(Image.open(gt_mask_path))
  # Assuming alpha channel indicates the mask
  groundTruth = gt_mask_raw[:, :, 3] > 0
  groundTruth = groundTruth.astype(np.uint8)

  #crop ground truth to remove whitespace
  kernel = np.ones((5, 5), np.uint8)
  groundSmooth = cv2.erode(groundTruth,kernel,iterations = 2)
  coords = np.argwhere(groundSmooth != 0)

  if coords.size > 0: # Only crop if there are non-zero pixels
    y_min, x_min = coords.min(axis=0)
    y_max, x_max = coords.max(axis=0)
    #print(coords) # Debug print, can be removed
    groundTruth = groundTruth[y_min:y_max+1, x_min:x_max+1]

  # Get predicted masks from modelData
  predicted_masks_data = modelData[i].get("masks", [])

  # If predicted_masks_data is a tuple, take the first element (as observed in modelData structure)
  if isinstance(predicted_masks_data, tuple):
      predicted_masks_data = predicted_masks_data[0]

  # Initialize prediction mask as zeros with the same shape as groundTruth
  prediction = np.zeros(groundTruth.shape[:2], dtype=np.uint8)

  # Combine all predicted masks for the current image
  # Check if there are any predicted masks (predicted_masks_data can be an empty array if no masks are detected)
  if predicted_masks_data.shape[0] > 0:
      for pred_mask in predicted_masks_data:
          # Ensure pred_mask is 2D, handling potential extra dimensions from SAM2 output
          while len(pred_mask.shape) > 2:
              pred_mask = pred_mask[0]

          # Resize predicted mask to match ground truth size
          pred_mask_resized = cv2.resize(pred_mask,
                                          (groundTruth.shape[1], groundTruth.shape[0]),
                                          interpolation=cv2.INTER_NEAREST)
          # Convert to binary mask
          pred_mask_binary = (pred_mask_resized > 0).astype(np.uint8)
          prediction = np.logical_or(prediction, pred_mask_binary).astype(np.uint8)

  # Apply cropping to the combined prediction mask *after* all masks are merged
  predSmooth = cv2.erode(prediction,kernel,iterations = 1)
  coords = np.argwhere(predSmooth != 0)
  if coords.size > 0: # Only crop if there are non-zero pixels
    y_min, x_min = coords.min(axis=0)
    y_max, x_max = coords.max(axis=0)
    prediction = prediction[y_min:y_max+1, x_min:x_max+1]
  prediction = cv2.resize(prediction, (groundTruth.shape[1], groundTruth.shape[0]), interpolation=cv2.INTER_NEAREST)
  # If no masks predicted, prediction remains all zeros

  # --- Calculate IoU ---
  intersection = np.logical_and(prediction, groundTruth).sum()
  union = np.logical_or(prediction, groundTruth).sum()

  iou = intersection / union if union > 0 else 0.0
  all_ious.append(iou)
  print(f"IoU for {filename}: {iou:.4f}")

  # --- Calculate Dice Coefficient ---
  # Dice = 2 * Intersection / (Sum of areas)
  sum_of_areas = prediction.sum() + groundTruth.sum()
  dice_coeff = (2. * intersection) / sum_of_areas if sum_of_areas > 0 else 0.0
  all_dice_coeffs.append(dice_coeff)
  print(f"Dice Coefficient for {filename}: {dice_coeff:.4f}")

  # --- Calculate Pixel Accuracy (PA) ---
  correct_pixels = (prediction == groundTruth).sum()
  total_pixels = groundTruth.size
  pixel_accuracy = correct_pixels / total_pixels if total_pixels > 0 else 0.0
  all_pixel_accuracies.append(pixel_accuracy)
  print(f"Pixel Accuracy for {filename}: {pixel_accuracy:.4f}")

  # --- Save overlaped mask image ---
  blank_image = np.zeros((groundTruth.shape[0], groundTruth.shape[1], 3), dtype=np.uint8)
  image = blank_image.copy()

  # Create 3-channel color mask for prediction
  colorMaskPred = np.zeros_like(image) # Now colorMaskPred is (H, W, 3)
  colorMaskPred[:, :, 1] = prediction * 255  # Assign to green channel
  image = cv2.addWeighted(image, 1.0, colorMaskPred, 0.5, 0)

  # Create 3-channel color mask for ground truth
  colorMaskTrue = np.zeros_like(image) # Now colorMaskTrue is (H, W, 3)
  colorMaskTrue[:, :, 2] = groundTruth * 255 # Assign to red channel
  image = cv2.addWeighted(image, 1.0, colorMaskTrue, 0.5, 0)

  cv2.imwrite(f"/content/Validation/Overlaps/{filename}", image)


# Calculate average and standard deviation for metrics
mean_iou = np.mean(all_ious) if all_ious else 0.0
std_iou = np.std(all_ious) if all_ious else 0.0
mean_dice = np.mean(all_dice_coeffs) if all_dice_coeffs else 0.0
std_dice = np.std(all_dice_coeffs) if all_dice_coeffs else 0.0
mean_pixel_accuracy = np.mean(all_pixel_accuracies) if all_pixel_accuracies else 0.0

print(f"\nSummary of Segmentation Metrics:")
print(f"Mean IoU: {mean_iou:.4f}, Std IoU: {std_iou:.4f}")
print(f"Mean Dice Coefficient: {mean_dice:.4f}, Std Dice Coefficient: {std_dice:.4f}")
print(f"Mean Pixel Accuracy: {mean_pixel_accuracy:.4f}")